# NSTT — Whisper-Small Fine-Tuning (T-003)

Fine-tune [`openai/whisper-small`](https://huggingface.co/openai/whisper-small) on SLR54 manifests from T-002.

**Before running:**
1. Upload the full `NSTT/` folder to Google Drive (or clone the repo in Colab).
2. Run `notebooks/00_setup.ipynb` — GPU runtime + dependencies.
3. Run `notebooks/01_data_prep.ipynb` for full SLR54 manifests (~8 GB). Synthetic manifests work for smoke tests only.

Checkpoints save under `models/whisper-small-ne/` on Drive so training survives disconnects.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/NSTT")
if not PROJECT_ROOT.exists():
    PROJECT_ROOT = Path("/content/NSTT")

sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)
print("Project root:", PROJECT_ROOT)

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No GPU — Runtime → Change runtime type → T4 GPU")
print(f"CUDA: {torch.cuda.get_device_name(0)}")

In [ ]:
OUTPUT_DIR = PROJECT_ROOT / "models" / "whisper-small-ne"
MANIFEST_DIR = PROJECT_ROOT / "data" / "manifests"

SMOKE_TEST = False   # True → 6 steps for sanity check
SEED = 42
RESUME = None        # e.g. OUTPUT_DIR / "checkpoint-500" after disconnect

print("Manifests:", MANIFEST_DIR)
print("Checkpoints:", OUTPUT_DIR)

In [ ]:
from src.training import create_trainer

trainer, processor = create_trainer(
    PROJECT_ROOT,
    OUTPUT_DIR,
    smoke_test=SMOKE_TEST,
    seed=SEED,
    manifest_dir=MANIFEST_DIR,
)

print("Model:", trainer.model.config._name_or_path)
print("max_steps:", trainer.args.max_steps)
print("eval_steps:", trainer.args.eval_steps)
print("batch:", trainer.args.per_device_train_batch_size,
      "x grad_accum", trainer.args.gradient_accumulation_steps)

In [ ]:
train_result = trainer.train(resume_from_checkpoint=RESUME)
eval_metrics = trainer.evaluate()

trainer.save_model(str(OUTPUT_DIR))
processor.save_pretrained(str(OUTPUT_DIR))

print("Train loss:", train_result.training_loss)
print("Eval:", eval_metrics)

In [ ]:
from transformers import WhisperForConditionalGeneration, WhisperProcessor

WhisperForConditionalGeneration.from_pretrained(OUTPUT_DIR)
WhisperProcessor.from_pretrained(OUTPUT_DIR)
print("Checkpoint reload OK →", OUTPUT_DIR)

In [ ]:
# Optional: quick inference on one val utterance
import soundfile as sf
from src.manifests import read_jsonl_manifest

row = read_jsonl_manifest(MANIFEST_DIR / "val.jsonl")[0]
audio, sr = sf.read(PROJECT_ROOT / row["audio_path"])

model = WhisperForConditionalGeneration.from_pretrained(OUTPUT_DIR).to("cuda")
proc = WhisperProcessor.from_pretrained(OUTPUT_DIR)

inputs = proc(audio, sampling_rate=sr, return_tensors="pt").to("cuda")
pred_ids = model.generate(**inputs, language="nepali", task="transcribe")
hypothesis = proc.batch_decode(pred_ids, skip_special_tokens=True)[0]

print("Reference:", row["transcript"])
print("Hypothesis:", hypothesis)